# Reading and Writing Binary Records

In this lesson, you will learn to save and restore primitive values using a shared binary format and diagnose incomplete records.

CSC-239 · Module 11 · Lesson 1 of 3

A campus event desk needs to save a seat offer and restore its count, price, and booking status with the same meanings. You will move from matching typed reads and writes to testing a record that is incomplete or interpreted incorrectly.

Use your earlier knowledge of paths, isolated file fixtures, resource closing, and exception handling. Each complete example creates and removes its own temporary file. The [Module 11 glossary](terms.md) supports the definitions taught beside the code.


## Learning Goals

- Write and restore primitive records following a documented type/order/meaning schema.
- Distinguish a same-type meaning error from an incomplete required field and verify cleanup.


## Why This Matters

Stored data is useful only when another part of a program can recover its meaning. A seat count and a room number may both be integers, yet confusing them could send a booking report to the wrong room. Checking that a program ran without an exception would miss that mistake.

A shared binary format gives the writer and reader an explicit agreement about the stored values. It also lets the program reject a record that ends before a required value is present. These ideas prepare you to follow an application's supplied input/output interface and file format. The private formats in this lesson are practice examples; they do not replace the supplied project interface or shared classes.


## Check Your Starting Point

Connect the file and resource ideas you have already used before adding typed binary operations.


Recall why a Path identifies a location rather than opening a file. Explain what try-with-resources closes and how an outer finally block can remove an owned file after an error. Contrast a collection-processing stream from Module 10 with file input/output.


In [ ]:
Your response:

Record your reasoning here.


<details>
<summary>Show answer</summary>

A Path value identifies a location. It does not open a file or prove that a file exists. An operation on Files opens or changes the entry at that location.

Try-with-resources closes the resources declared in its parentheses as control leaves that block. An outer finally block can then remove the example's own file whether the attempted work succeeds or fails. Closing a stream and deleting its file are separate actions.

The Module 10 collection-processing stream describes operations over elements already in a collection. A file input/output stream carries data to or from a file. The shared word does not make their operations interchangeable.

</details>


## Video Demonstration

Follow a seat offer through a typed writer and matching reader. The demonstration connects the stored field order to the named values recovered by the program.

<video controls preload="metadata" width="960">
  <source src="media/01_reading_and_writing_binary_records/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/01_reading_and_writing_binary_records/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the binary record demonstration transcript](media/01_reading_and_writing_binary_records/transcript.md).


## Concept

### Store values as bytes with an agreed interpretation

A campus event desk needs to save a small seat record and restore it later. The record contains four available seats, a price of $2.50 per seat, and a Boolean indicating that booking is open. These are three different kinds of values with three different meanings. Success means restoring all three meanings accurately, not merely reading some bytes without an error.

A **byte** is a unit of eight bits. A **byte stream** transfers a sequence of bytes into or out of a program. A **binary file** stores bytes that an application interprets according to a chosen format. Text files also store bytes, but the earlier lessons decoded those bytes as characters. This lesson uses representations of primitive values instead.

The `.bin` suffix describes our intended use of the file; it does not make Java choose a format automatically. The writer and reader operations determine how the bytes are stored and interpreted. Opening this record in a text editor is not the same as decoding its integer, double, and Boolean fields.

We will use two layers: a file stream moves bytes, and a data stream supplies operations for writing or reading specific Java primitive types. Keeping those roles separate explains the nested constructors in the complete example.

### Write each field through its matching operation

**Typed binary output** writes a value using the representation associated with its type. A **DataOutputStream** provides methods such as `writeInt`, `writeDouble`, and `writeBoolean` for this purpose. It wraps the byte output stream opened by **`Files.newOutputStream`**.

```java
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(4);
        writer.writeDouble(2.5);
        writer.writeBoolean(true);
    }
```

This fragment assumes the temporary Path named `file` and the imports from the complete program. The inner `Files.newOutputStream` call opens byte output for that file. The outer constructor supplies the typed writing operations. The try-with-resources declaration closes the writer and its underlying stream when the block exits.

The first call writes the available-seat count as an int. The second writes the price as a double. The third writes the booking flag as a Boolean value. Their order becomes part of the stored record. These calls do not write field names or printable lines such as `Seats: 4`.

With the default output options, an existing regular file is opened for writing and its earlier contents are truncated. Our example owns a newly created temporary file, so the record does not depend on bytes left by a previous exercise. The writer's block finishes before the reader opens the completed record.

### Read the fields in the corresponding order

**Typed binary input** consumes bytes using a requested primitive representation. A **DataInputStream** wraps the byte input stream opened by **`Files.newInputStream`**. The corresponding read calls restore values for our variables:

```java
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
```

This fragment belongs inside the reader's try-with-resources block in the complete program. `readInt` consumes the next integer representation and returns its value for `seats`. `readDouble` then consumes the price representation. `readBoolean` reads the booking flag. The stream's position advances after each operation; the next read does not start again at the beginning.

The read methods do not search for variable names. Choosing the name `price` cannot tell the stream which bytes hold a price. Our call order must match the record format chosen by the writer.

Once restored, the values can be printed as ordinary text. The later lines `Seats: 4`, `Price: 2.5`, and `Open: true` are console descriptions of the decoded values, not evidence that those same character strings were stored in the binary file.

### Follow one complete count record

This full reading example uses the same typed operations for one count of six. The imports supply the four short library names. `Files.createTempFile` creates a new empty temporary file and returns its Path. Here `count-example-` is the requested name prefix and `.bin` is its suffix; generated characters give the new file its own name. This differs from merely constructing a Path value.

The program writes one int, closes the writer, and then opens its reader. The outer finally removes only this example's file. This is an explained example, so its expected result is shown openly.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("count-example-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(6);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        System.out.println("Count: " + reader.readInt());
    }
} finally {
    Files.deleteIfExists(file);
}
```

The output is `Count: 6`. The value passed to writeInt is recovered by readInt at the start of the new reader. The print label is added only at the console. Both resource blocks close before finally removes the owned file. The complete program can be copied into a Java work cell to inspect it; its imports and fixture setup are all included.


### Treat types, order, and meaning as one schema

A **binary record schema** is the agreement that assigns types, positions, and meanings to the stored fields. **Field order** specifies which field comes first, second, and so on. Our schema is:

| Position | Java type | Meaning | Example value |
| --- | --- | --- | --- |
| First | int | Available seats | 4 seats |
| Second | double | Price per seat | 2.5 dollars |
| Third | boolean | Booking is open | true |

The writer and reader must follow all three parts of that agreement. Reading with the wrong type can consume the wrong number of bytes or interpret a representation incorrectly. It is not a reliable way to ask Java to discover the stored type.

Matching types alone is also insufficient. Suppose a different record stores two integers: seats available followed by seats requested. A reader that assigns the first integer to requested seats and the second to available seats can complete both reads while reversing their meanings. No type mismatch is required for that mistake.

This is why a documented schema and a meaningful test belong together. Check that restored values represent the intended facts, not just that no exception occurred. The guided examples will contrast a meaning error with an incomplete record, which is a different problem.

### Inspect a complete meaning error

This complete diagnostic writes room number 12 followed by available count 3. Its reader is intentionally wrong: it assigns the first int to available and the second to room. The example makes that distinction visible without changing either stored value.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("wrong-order-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int available = reader.readInt();
        int room = reader.readInt();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
    }
} finally {
    Files.deleteIfExists(file);
}
```

It prints `Room: 3` and `Available: 12`. Both reads finish because each consumes a complete int. The successful reads do not check the application meaning of either position. Assigning room first and available second would match the writer's agreement; changing the print labels alone would hide the faulty assignments rather than fix them.


Follow primitive fields from typed writes through a closed writer to matching typed reads. The sequence uses the explained values above. The labeled field boxes show meaning and order, not a literal byte dump.

<details class="animation-panel" open>
<summary>Show or hide animation: Write and restore the agreed fields</summary>

<p><img src="media/01_reading_and_writing_binary_records/typed_record_round_trip.gif" alt="Three labeled fields are written, then read in the same order and restored as 4 seats, a price of 2.5, and an open status of true." width="960" style="max-width:100%;height:auto;"></p>

</details>

[View still: Write and restore the agreed fields](media/01_reading_and_writing_binary_records/typed_record_round_trip_still.png). The sequence lasts 12.5 seconds; hide it to stop visible motion. The prose and still preserve the explanation.


Show that successful same-type reads can assign plausible values to the wrong meanings. The sequence uses the explained values above. The labeled field boxes show meaning and order, not a literal byte dump.

<details class="animation-panel" open>
<summary>Show or hide animation: Read complete values with swapped meanings</summary>

<p><img src="media/01_reading_and_writing_binary_records/same_type_wrong_meaning.gif" alt="Two int values retain their positions while swapped reader assignments produce Room: 3 and Available: 12." width="960" style="max-width:100%;height:auto;"></p>

</details>

[View still: Read complete values with swapped meanings](media/01_reading_and_writing_binary_records/same_type_wrong_meaning_still.png). The sequence lasts 10 seconds; hide it to stop visible motion. The prose and still preserve the explanation.


### Distinguish a missing required field from a valid value

An **incomplete binary record** ends before a required field can be read completely. The data input methods used here report **EOFException**, a kind of IOException, when they reach the end of input without enough bytes for that required value. EOF means end of file.

A required Boolean field that is missing is not the same as a stored `false`. The first case means the record did not supply the value promised by its schema; the second is a valid value with a meaning such as booking closed. Returning a default value after ignoring the exception would hide that distinction.

The incomplete-record demonstration later deliberately writes less than its documented format requires, then attempts the required read. Its exception explains why the reader cannot report a complete record. The preceding field values may have been read successfully, but that does not make the whole record complete.

The examples use try-with-resources for their streams and an outer finally block for the owned temporary file. Closing streams releases their resources. The finally block attempts to remove the file even if a read fails. These are separate cleanup responsibilities, and a cleanup operation can itself fail; they do not turn an incomplete record into a valid one.

We now have the rules needed for the full example: establish the schema, write all required fields, close output, read corresponding fields, interpret the restored values, and clean up the owned file. Later lessons build on this foundation to store object state.

### Follow a required read that cannot finish

This complete comparison keeps room 12 and available count 3 but omits the required open flag. The reader still requires int, int, boolean. Its handler catches EOFException so we can observe a controlled failure report.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("incomplete-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Complete record.");
    } catch (EOFException exception) {
        System.out.println("Incomplete room record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

The output is `Incomplete room record.` The two ints are available, but readBoolean cannot finish. Control leaves the reader's resource block, closing the reader, and reaches the matching catch. The success print is skipped because it follows the failed read. Finally then removes the owned file. No false flag is invented.


Trace a missing required boolean to EOFException and skipped complete-record output. The sequence uses the explained values above. The labeled field boxes show meaning and order, not a literal byte dump.

<details class="animation-panel" open>
<summary>Show or hide animation: Reach a missing required field</summary>

<p><img src="media/01_reading_and_writing_binary_records/missing_required_field.gif" alt="A reader reaches a missing required boolean, skips the complete message and prints Incomplete room record." width="960" style="max-width:100%;height:auto;"></p>

</details>

[View still: Reach a missing required field](media/01_reading_and_writing_binary_records/missing_required_field_still.png). The sequence lasts 12.5 seconds; hide it to stop visible motion. The prose and still preserve the explanation.


## Worked Example

### Create one location for the seat offer

The event desk offers **4 seats** at **$2.50 per seat**, and booking is **open**. We will save those three primitive values, close the writer, and restore them for a report. The format is int for seats, double for price, then boolean for booking status.

The following excerpts belong to the complete runnable program below. The four imports make the two data-stream types, Files, and Path available by their short names.

```java
Path file = Files.createTempFile("seat-record-", ".bin");
```

As in the count example, `createTempFile` creates a new empty temporary file and returns its Path. The prefix helps identify its purpose; generated characters distinguish the new name, and the suffix supplies `.bin`. Store that returned location in `file` so writing, reading, and deletion all use the same file. The suffix itself does not set the format.

### Store the three agreed fields

```java
try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
    writer.writeInt(4);
    writer.writeDouble(2.5);
    writer.writeBoolean(true);
}
```

The inner Files call opens byte output at `file`. The DataOutputStream constructor wraps it with primitive-writing operations. The resource variable `writer` is available inside this block, and try-with-resources closes it when the block ends.

The three statements write the seat count, price, and status in their agreed order. They do not store the names seats, price, and open. Once this block ends, the writer is closed before the next block opens a reader.

### Restore values before reporting them

```java
int seats = reader.readInt();
double price = reader.readDouble();
boolean open = reader.readBoolean();
```

In the complete program, DataInputStream wraps the byte input opened by Files.newInputStream. Its new reader starts at the beginning. Each declaration receives the next value using the operation that matches its written type. The variable's name and later use carry its meaning in the report.

Each println combines a label with the corresponding restored variable. Those labels are console text; they were not stored in the file. All three reads precede these prints, so the report starts only after all required fields have been restored.

### Close resources and remove the owned file

```java
} finally {
    Files.deleteIfExists(file);
}
```

This is the closing part of the outer try in the full program. The reader's resource block closes before control reaches finally. The deletion call removes this program's temporary file if it is present; it does not replace resource closing. It can itself fail, so these examples do not promise cleanup under every possible file-system failure.

The Java notebook permits these checked I/O calls at the top level. In a conventional Java method, handle or declare checked exceptions as in Module 7. The video places the same demonstration inside a main method with a throws declaration.


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(4);
        writer.writeDouble(2.5);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}


Expected output:

```text
Seats: 4
Price: 2.5
Open: true
```

The recovered report offers four seats at a price of 2.5 per seat, with booking open. Java prints this double as `2.5`; currency formatting is outside this example. Matching write and read operations recover each value, while their positions and assignments preserve the meanings.

The reader closes and finally removes the private file after the report. Running this complete cell again creates a new temporary file; it does not depend on the deleted one.


## Guided Practice

Use the same operations with different values, then reduce the support by completing, changing, and repairing a program. Keep the answers closed while making each attempt. Blank Java work cells are for your complete programs; the separate writing cells hold predictions and explanations.


### Predict a different seat offer

Read the complete program below without running it. Predict every printed line and pair each read with the write that supplies its value. Explain how each position gets its field meaning and why the writer closes before the reader opens.


In [ ]:
Your response:

Record your reasoning here.


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}


Run the complete prediction program. Keep your original prediction and record all actual lines below. Explain any correction by tracing the matching write and read.


In [ ]:
Your response:

Record your reasoning here.


Trace three field rows: position, meaning, write operation and value, and matching read operation. Explain the separate jobs of Files.newOutputStream, DataOutputStream, Files.newInputStream, and DataInputStream. Trace writer close, reader open, reader close, and file deletion. Would renaming a reader variable change stored bytes? Replay the entire program and explain why its file setup is independent of the earlier run.


In [ ]:
Your response:

First field:
Second field:
Third field:
Raw streams and wrappers:
Closing and deletion order:
Renaming and fresh replay:


<details>
<summary>Show answer</summary>

The writer stores an int with value 9, a double with value 1.75 and a boolean with value false, in that order. The reader uses readInt, readDouble and readBoolean in the same order and assigns each restored value to its intended meaning. It prints Seats: 9, Price: 1.75 and Open: false.

A read consumes the next encoded value; it does not search for the variable name. The writer closes before the reader opens. The reader closes before finally deletes this program’s own temporary file.

Replaying the whole program creates a new temporary file. Files opens raw byte access. The data-stream wrappers supply operations for primitive values.

Renaming a variable does not rewrite the file or move the reader’s position; the assignment and later use determine the program’s interpretation.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 9
Price: 1.75
Open: false
```

Common error: Reading the fields in the order of their print statements without checking the actual read order. Assuming a variable name is stored as a field label. Treating a .bin extension as a check of the stored format.

</details>


### Distinguish a missing flag from stored false

The complete program below deliberately omits the final boolean write while retaining all required reads. Its EOFException handler is already supplied. Before running, predict the first read that cannot finish and whether any named field lines will print.


In [ ]:
Your response:

Record your reasoning here.


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete seat record.");
    }
} finally {
    Files.deleteIfExists(file);
}


Run the missing-flag program. Record the actual output and compare it with your prediction. Identify the failed read and explain why later statements do or do not execute.


In [ ]:
Your response:

Record your reasoning here.


Now prepare an empty-file comparison in the same work program: remove both remaining write statements but retain the empty writer block, all reads, the catch, and finally cleanup. Before running this complete variant, predict its output and first failing read.


In [ ]:
Your response:

Prediction before the empty-file run:
First failing read:


Run the empty-file variant. Record its actual output and first failing read. Compare it with the missing-flag case, explaining why the same handler message can describe different failed reads. Explain why absence is different from a stored false value and how the resources and owned file are cleaned up.


In [ ]:
Your response:

Record your reasoning here.


<details>
<summary>Show answer</summary>

The two stored values can be read, but the required boolean has not been written. readBoolean throws EOFException. All named print statements follow all three reads, so none executes. The catch prints Incomplete seat record.

This is a handled incomplete-record failure, not a successful complete record or a restored false flag. The reader closes as control leaves its resource block, and finally deletes the temporary file. In the empty-file comparison, readInt is the first failing read.

The same catch message therefore describes a different point of failure; use the written schema and the source to locate it.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete seat record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Incomplete seat record.
```

Common error: Substituting false for a flag that was never read. Claiming the earlier field print statements run even though they appear after the failed read. Using an identical catch message as proof that the same read failed.

**Check case 2.** No first int is available, so readInt fails before any later read. The catch message matches the missing-flag case even though the first missing field differs. The resource blocks still close and finally removes the private file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete seat record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Incomplete seat record.
```

</details>


### Complete typed wrappers and price operations

The template below needs four replacements: OUTPUT_WRAPPER, INPUT_WRAPPER, WRITE_PRICE, and READ_PRICE. Choose between DataOutputStream and DataInputStream for the wrappers, and writeDouble and readDouble for the price calls. Explain your choices and reconstruct the expected output of this already-seen data set before running. Then copy the complete repaired program into the Java work cell.

This template is intentionally incomplete:

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new OUTPUT_WRAPPER(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.WRITE_PRICE(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new INPUT_WRAPPER(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.READ_PRICE();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```


In [ ]:
Your response:

Record your reasoning here.


Run your completed program. Record the actual output, compare it with your reconstruction, and explain how each replacement matches its input/output direction and field type.


In [ ]:
Your response:

Record your reasoning here.


<details>
<summary>Show answer</summary>

Use DataOutputStream for OUTPUT_WRAPPER, DataInputStream for INPUT_WRAPPER, writeDouble for WRITE_PRICE and readDouble for READ_PRICE. Files opens the raw byte stream; the matching wrapper supplies typed operations. The price is written and read as double.

The writer stores an int with value 9, a double with value 1.75 and a boolean with value false, in that order. The reader uses readInt, readDouble and readBoolean in the same order and assigns each restored value to its intended meaning. It prints Seats: 9, Price: 1.75 and Open: false.

A read consumes the next encoded value; it does not search for the variable name. The writer closes before the reader opens. The reader closes before finally deletes this program’s own temporary file.

Replaying the whole program creates a new temporary file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 9
Price: 1.75
Open: false
```

Common error: Choosing the input wrapper for writing or the output wrapper for reading. Using readInt for the price field. Leaving a placeholder in the executable program.

</details>


### Change both sides of the agreement

First run the unchanged starter below and record its output. Plan a revised schema that writes price before seats, followed by open status. Change the first two reader declarations to follow that order. Keep all stored values, print labels, and print order unchanged. Before running your modified program, state the revised field agreement and predict its named output.


In [ ]:
Your response:

Unchanged starter output:
Revised field agreement:
Prediction before the modified run:


In [ ]:
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(9);
        writer.writeDouble(1.75);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int seats = reader.readInt();
        double price = reader.readDouble();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}


Run the complete modified program. Record its actual output and explain why the named meanings remain the same even though the stored field order changed. Identify the corresponding changes made to both writer and reader.


In [ ]:
Your response:

Record your reasoning here.


Keep your revised double, int, boolean order. Change the values to zero price, zero seats, and true open status. Predict the exact printed lines before running this complete variant.


In [ ]:
Your response:

Record your reasoning here.


Run the zero-value variant. Record and explain its actual output, distinguishing explicit zero and true values from a missing required field.


In [ ]:
Your response:

Record your reasoning here.


<details>
<summary>Show answer</summary>

The revised agreement is double, int, boolean: price first, seats second and open status third. Both the write order and the read order change together. The print statements still use the correctly assigned seats, price and open variables, so the named output remains Seats: 9, Price: 1.75 and Open: false.

Changing only the print order would not repair a disagreement between writer and reader. The additional complete test restores zero seats, price 0.0 and true; these are actual stored values rather than replacements for missing fields.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeDouble(1.75);
        writer.writeInt(9);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        double price = reader.readDouble();
        int seats = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 9
Price: 1.75
Open: false
```

Common error: Changing the writer order while leaving the reader unchanged. Moving only print statements instead of matching the read sequence. Confusing a stored zero or true value with absence of a field.

**Additional test: `Revised double,int,boolean agreement with seats 0, price 0.0, open true`.** The same revised sequence restores explicit zero numeric values and true status. Every required field exists. The reader does not invent values when input is absent.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("seat-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeDouble(0.0);
        writer.writeInt(0);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        double price = reader.readDouble();
        int seats = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Seats: " + seats);
        System.out.println("Price: " + price);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Seats: 0
Price: 0.0
Open: true
```

</details>


### Repair swapped meanings

The diagnostic below writes room number 25, available count 2, and open true. Its reader assigns the first two fields to the wrong meanings. Without running it, predict its named output and explain why its reads can complete without EOFException. Repair only those two reader declarations in a complete copy placed in the Java work cell.

This program is intentionally wrong about field meanings:

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(25);
        writer.writeInt(2);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int available = reader.readInt();
        int room = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```


In [ ]:
Your response:

Record your reasoning here.


Run your repaired program. Record the two repaired declarations and all actual output. Explain how the assignments restore the writer's intended meanings and why equal file length would not detect the original mistake.


In [ ]:
Your response:

Record your reasoning here.


<details>
<summary>Show answer</summary>

The faulty program completes its reads and prints Room: 2, Available: 25 and Open: true. Each readInt consumes one complete int, but the first is assigned to available and the second to room. Both numeric reads use the same type and encoded size, so enough bytes exist and EOFException is not expected for this file.

Its byte length also stays unchanged. The repair reads room first and available second, matching the writer’s meanings as well as its types. It prints Room: 25, Available: 2 and Open: true.

This is a meaning error that normal completion cannot detect; compare each named field with the intended schema.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(25);
        writer.writeInt(2);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 25
Available: 2
Open: true
```

Common error: Changing the writer to conceal a reader that assigns meanings incorrectly. Assuming that two int fields are interchangeable because their types match. Treating normal completion or an unchanged byte length as proof of the schema.

</details>


## Independent Practice

### Build and test a room-availability record

A campus room desk needs the room number, available-seat count, and open status restored with their original meanings. Write your own complete program using the format described below.


Create a private temporary binary file containing room number 12, available count 3, and open false. Use int, int, boolean in that order for writing and reading. The report must name Room, Available, and Open on separate lines in that order. Include every import and all setup, close both streams, and delete the private file. Before coding and running, document the three field positions and predict the exact output in the writing cell. Then construct your complete program in the Java cell.


In [ ]:
Your response:

Record your reasoning here.


Run your complete baseline program. Record all actual output and explain how the reader restores each named field. Identify where the writer and reader close and where the temporary file is deleted.


In [ ]:
Your response:

Record your reasoning here.


### Test zero available seats

Keep a copy of your baseline. Make a complete variant for room 12, available 0, and open false. Write your prediction below before its run. Run the whole variant so it creates and cleans up its own temporary file, then record the actual lines and explain the zero value.


In [ ]:
Your response:

Prediction before run:
Actual output after run:
Explanation:


### Test true status

Make another complete variant for room 12, available 3, and open true. Write your prediction before running it. Run the whole program with fresh file setup and cleanup, then record and explain its actual output.


In [ ]:
Your response:

Prediction before run:
Actual output after run:
Explanation:


### Test an omitted required flag

Restore the baseline numeric writes, but omit the boolean write. Retain all three required reads and put every named-field print after them. Add the EOFException import and a catch immediately after the reader's resource block that prints `Incomplete room record.` Keep the outer finally cleanup. Before running, predict the first failing read and whether any named lines print. Run the complete variant, then record the actual result and explain its control flow.


In [ ]:
Your response:

Prediction and first failing read before run:
Actual output after run:
Control-flow explanation:


Compare your baseline, zero-available, true-status, and omitted-flag tests. Explain why missing is different from false; why swapping the two int meanings can yield plausible wrong values; and why equal file lengths are insufficient. Identify resource closing and cleanup in each full test, and explain why each test creates its own file.


In [ ]:
Your response:

Baseline:
Zero available:
True status:
Omitted flag:
Meaning, length, and cleanup comparison:


<details>
<summary>Show answer</summary>

The writer stores room number 12, available count 3 and false in the required int, int, boolean order. The reader restores those fields in the same order and prints Room: 12, Available: 3 and Open: false. The writer closes before reading, and the reader closes before finally deletes the private temporary file.

Both int reads have the same operation and encoded size, but their positions still have different meanings. Swapping their target meanings could complete without EOFException while labeling the values incorrectly. The zero-available and true-status tests each contain a complete record and restore the exact supplied value.

The missing-flag test contains both ints but lacks the required boolean. readBoolean throws EOFException before any named print executes; the handler reports Incomplete room record. Neither false nor a default record is substituted. Zero, false and true are valid stored values; an absent required field is a different condition.

A meaning check compares the named values with their intended positions, while the incomplete-input test checks whether the required data is present.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 12
Available: 3
Open: false
```

Common error: Reading available before room even though room was written first. Reusing an already deleted temporary path without recreating the complete fixture. Omitting imports or setup and depending on an earlier notebook.

**Additional test: Room 12, zero available seats, false status.** The zero count was written as an int and restored from a complete record. It is a valid stored value, not a missing-field substitute.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(0);
        writer.writeBoolean(false);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 12
Available: 0
Open: false
```

**Additional test: Room 12, available 3, true status.** Only the stored boolean changes from the baseline; the reader restores true through the same required readBoolean call.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
        writer.writeBoolean(true);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Room: 12
Available: 3
Open: true
```

**Additional test: Baseline numeric fields written; final flag omitted and EOFException caught.** Both ints are present, but readBoolean cannot finish. No named record lines print because all three prints follow that read. The catch reports an incomplete room record, the reader closes and finally deletes the private file.

```java
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.EOFException;
import java.nio.file.Files;
import java.nio.file.Path;
Path file = Files.createTempFile("room-record-", ".bin");
try {
    try (DataOutputStream writer = new DataOutputStream(Files.newOutputStream(file))) {
        writer.writeInt(12);
        writer.writeInt(3);
    }
    try (DataInputStream reader = new DataInputStream(Files.newInputStream(file))) {
        int room = reader.readInt();
        int available = reader.readInt();
        boolean open = reader.readBoolean();
        System.out.println("Room: " + room);
        System.out.println("Available: " + available);
        System.out.println("Open: " + open);
    } catch (EOFException problem) {
        System.out.println("Incomplete room record.");
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Incomplete room record.
```

</details>


## Summary

Byte streams carry data; typed data streams supply operations for primitive values. A record schema connects each stored position with a type and meaning. Both writer and reader must follow that agreement.

Two int reads can succeed while assigning the wrong meanings. An EOFException during a required read is a different problem: the record lacks needed input. Check named values, complete boundary cases, and incomplete input in separate fresh fixtures. Close each stream and remove only the file owned by that test.


Close the answers. Explain how you distinguish a successful round trip, a same-type meaning mismatch, and an incomplete required field. State the observation you would check for each.


In [ ]:
Your response:

Record your reasoning here.


<details>
<summary>Show answer</summary>

A successful round trip restores every required value with its intended type, position, and meaning. Compare the named results with the writer's field agreement.

A meaning mismatch may still complete all reads. For example, two int values assigned to the wrong meanings can print plausible but incorrect labels. Equal byte lengths do not expose that mistake.

An incomplete record lacks bytes needed by a required read. That read throws EOFException; a stored false or zero would instead be a value recovered by a successful read. Both kinds of tests still need resource closing and owned-file cleanup.

</details>


## Reflection

Transfer the format agreement to a record used in a subject or activity you know. Then consider the next lesson: it groups supported object state for writing and restoring, instead of having you manually write every primitive field. An agreement between the writing and reading code will still matter.


Design three fields for a record from a subject or activity you know. State their meanings, types, and storage order. Describe a mistake that could produce plausible but wrongly labeled values. Choose a boundary-value test and a missing-field test, explaining what each checks.


In [ ]:
Your response:

Record your reasoning here.


## Supplemental Reading

- [DataOutputStream primitive-writing operations](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/DataOutputStream.html) documents the typed writes used in these records.
- [DataInputStream primitive-reading operations](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/DataInputStream.html) documents matching reads and required input.
- [EOFException and incomplete input](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/EOFException.html) explains premature end-of-input failures.
- [Files temporary files, byte streams, and deletion](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Files.html) documents createTempFile, newInputStream, newOutputStream, and deleteIfExists.
